# Anomaly detection

Jeet Purohit 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import eli5
from lime.lime_tabular import LimeTabularExplainer

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

In [ ]:
df = pd.read_csv("loan_data.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Data types:")
print(df.dtypes)

In [ ]:
# Data distribution analysis

features = [
    'person_age', 'person_gender', 'person_education', 'person_income',
    'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
    'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
    'credit_score', 'previous_loan_defaults_on_file', 'loan_status'
]

categorical = [
    'person_gender', 'person_education', 'person_home_ownership',
    'loan_intent', 'previous_loan_defaults_on_file'
]
numerical = [f for f in features if f not in categorical]

for col in features:
    print(f"\n {col}")
    if col in numerical:
        print(f"  Min: {df[col].min()} | Max: {df[col].max()} | Mean: {df[col].mean():.2f} | Std: {df[col].std():.2f}")
    else:
        counts = df[col].value_counts()
        total = counts.sum()
        for val, cnt in counts.items():
            pct = cnt / total * 100
            print(f"  {val}: {cnt} ({pct:.1f}%)")

# Data Preproccessing

* Remove unnessesary features: Person_Gender
* The the means for each gender are not far apart, but we think that it is a possibility that the model will learn some bias from the data we have.

In [ ]:
mean_income_by_gender = df.groupby("person_gender")["person_income"].mean()
for gender, mean_income in mean_income_by_gender.items():
    print(f"{gender}: Mean income = {mean_income:.2f}")

* We want to change some of the categorical columns to Numerical values
* We also remove unnessesary features like "person_gender" beacuase this might not be needed

In [25]:
df = df.drop(columns=["person_gender"])

In [26]:
cat_cols = df.select_dtypes(include="object").columns.tolist()
print(f"Categorical columns: {cat_cols}\n")

Categorical columns: ['person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']



In [29]:
label_encoders = {}
df_numerical = df.copy()

for col in cat_cols:
    le = LabelEncoder()
    df_numerical[col] = le.fit_transform(df_numerical[col])
    label_encoders[col] = le
    # print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\nEncoded shape: {df_numerical.shape}")
df_numerical.head()


Encoded shape: (45000, 13)


,person_age,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,4,71948.0,0,3,35000.0,4,16.02,0.49,3.0,561,0,1
1,21.0,3,12282.0,0,2,1000.0,1,11.14,0.08,2.0,504,1,0
2,25.0,3,12438.0,3,0,5500.0,3,12.87,0.44,3.0,635,0,1
3,23.0,1,79753.0,0,3,35000.0,3,15.23,0.44,2.0,675,0,1
4,24.0,4,66135.0,1,3,35000.0,3,14.27,0.53,4.0,586,0,1


In [30]:
for col in cat_cols:
    le = LabelEncoder()
    le.fit(df[col])
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"{col} mapping:")
    for k, v in mapping.items():
        print(f"  {k}: {v}")
    print()

person_education mapping:
  Associate: 0
  Bachelor: 1
  Doctorate: 2
  High School: 3
  Master: 4

person_home_ownership mapping:
  MORTGAGE: 0
  OTHER: 1
  OWN: 2
  RENT: 3

loan_intent mapping:
  DEBTCONSOLIDATION: 0
  EDUCATION: 1
  HOMEIMPROVEMENT: 2
  MEDICAL: 3
  PERSONAL: 4
  VENTURE: 5

previous_loan_defaults_on_file mapping:
  No: 0
  Yes: 1

